Pip installs for Oracle 

In [2]:
# pip install gymnasium[atari]
# pip install gymnasium[accept-rom-license]
# pip install stable-baselines3

In [ ]:
import gymnasium as gym
import ale_py
import numpy as np
from stable_baselines3 import PPO

Grab the Oracle's weights and biases 

In [ ]:
class Pong6InputWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # 6 inputs: [P1_y, P2_y, Ball_x, Ball_y, V_x, V_y]
        self.observation_space = gym.spaces.Box(
            low=-255, high=255, shape=(6,), dtype=np.float32
        )
        self.prev_ball_pos = None

    def observation(self, obs):
        # Indices: P1_y: 51, P2_y: 50, Ball_x: 49, Ball_y: 54
        p1_y, p2_y = obs[51], obs[50]
        b_x, b_y   = obs[49], obs[54]

        # Velocity calculation
        if self.prev_ball_pos is None:
            v_x, v_y = 0, 0
        else:
            v_x = b_x - self.prev_ball_pos[0]
            v_y = b_y - self.prev_ball_pos[1]

        self.prev_ball_pos = (b_x, b_y)
        
        return np.array([p1_y, p2_y, b_x, b_y, v_x, v_y], dtype=np.float32)

    def reset(self, **kwargs):
        self.prev_ball_pos = None
        return super().reset(**kwargs)

--- Oracle Neural Network Architecture ---
ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=6, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)

--- Detailed Parameter Audit ---
mlp_extractor.policy_net.0.weight             | Weigh